In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/notebooks/puttadatta/fintech-project-2/__results__.html
/kaggle/input/notebooks/puttadatta/fintech-project-2/__notebook__.ipynb
/kaggle/input/notebooks/puttadatta/fintech-project-2/__output__.json
/kaggle/input/notebooks/puttadatta/fintech-project-2/highrisk_df.parquet
/kaggle/input/notebooks/puttadatta/fintech-project-2/custom.css


In [2]:
import pandas as pd
import sklearn

In [3]:
df = pd.read_parquet('/kaggle/input/notebooks/puttadatta/fintech-project-2/highrisk_df.parquet')
df.head()
df.tail()

,transaction_count,drawdown_rate,fraud_rate,drawdown_amp,fraud_amp,riskindx
312,153,0.019608,0.0,0.436035,0.0,0.436035
313,153,0.019608,0.0,0.436035,0.0,0.436035
314,131,0.015267,0.0,0.339508,0.0,0.339508
315,406,0.014778,0.0,0.328637,0.0,0.328637
316,180,0.011111,0.0,0.247087,0.0,0.247087


In [4]:
df.shape

(317, 6)

In [5]:
df.isna().sum()

transaction_count    0
drawdown_rate        0
fraud_rate           0
drawdown_amp         0
fraud_amp            0
riskindx             0
dtype: int64

In [6]:
risk_limit = df['riskindx'].quantile(.85)
print("the base risk limit is:",risk_limit)

the base risk limit is: 2.870397882986361


In [7]:
df['high_risk'] = (df['riskindx'] >= risk_limit).astype(int)
df.head()

,transaction_count,drawdown_rate,fraud_rate,drawdown_amp,fraud_amp,riskindx,high_risk
0,105,0.057143,0.019048,1.270732,14.532328,15.803059,1
1,172,0.063953,0.017442,1.422185,13.307219,14.729404,1
2,176,0.045455,0.017045,1.010809,13.004782,14.015591,1
3,137,0.043796,0.014599,0.973918,11.137916,12.111834,1
4,174,0.080460,0.011494,1.789249,8.769508,10.558757,1


In [8]:
df['high_risk_next'] = df['high_risk'].shift(-1)
df.head()

,transaction_count,drawdown_rate,fraud_rate,drawdown_amp,fraud_amp,riskindx,high_risk,high_risk_next
0,105,0.057143,0.019048,1.270732,14.532328,15.803059,1,1.0
1,172,0.063953,0.017442,1.422185,13.307219,14.729404,1,1.0
2,176,0.045455,0.017045,1.010809,13.004782,14.015591,1,1.0
3,137,0.043796,0.014599,0.973918,11.137916,12.111834,1,1.0
4,174,0.080460,0.011494,1.789249,8.769508,10.558757,1,1.0


In [9]:
df.shape

(317, 8)

In [10]:
df.tail()

,transaction_count,drawdown_rate,fraud_rate,drawdown_amp,fraud_amp,riskindx,high_risk,high_risk_next
312,153,0.019608,0.0,0.436035,0.0,0.436035,0,0.0
313,153,0.019608,0.0,0.436035,0.0,0.436035,0,0.0
314,131,0.015267,0.0,0.339508,0.0,0.339508,0,0.0
315,406,0.014778,0.0,0.328637,0.0,0.328637,0,0.0
316,180,0.011111,0.0,0.247087,0.0,0.247087,0,NaN


In [11]:
df = df.dropna(subset = ['high_risk_next'])
df.shape

(316, 8)

In [12]:
df['high_risk_next'].value_counts()

high_risk_next
0.0    269
1.0     47
Name: count, dtype: int64

In [13]:
df_model = df[['transaction_count','riskindx','high_risk_next']].copy()
df.isna().sum()

transaction_count    0
drawdown_rate        0
fraud_rate           0
drawdown_amp         0
fraud_amp            0
riskindx             0
high_risk            0
high_risk_next       0
dtype: int64

In [14]:
df_model.shape

(316, 3)

In [15]:
splite_index = int(len(df_model)*0.7)

train = df_model.iloc[:splite_index]
test = df_model.iloc[splite_index:]

X_train = train[['transaction_count','riskindx']]
y_train = train[['high_risk_next']]

X_test = test[['transaction_count','riskindx']]
y_test = test[['high_risk_next']]

In [16]:
ytrainpos = y_train['high_risk_next'].value_counts()
ytestpos = y_test['high_risk_next'].value_counts()
print("the shape of X_train:",X_train.shape)
print("the no of positives:",ytrainpos)
print("the shape of X_test:",X_test.shape)
print("the no of positives:",ytestpos)

the shape of X_train: (221, 2)
the no of positives: high_risk_next
0.0    174
1.0     47
Name: count, dtype: int64
the shape of X_test: (95, 2)
the no of positives: high_risk_next
0.0    95
Name: count, dtype: int64


In [17]:
df_model[df_model['high_risk_next'] == 1].index.max()

np.int64(46)

In [18]:
transition_events = (
    (df['high_risk'] == 0) & 
    (df['high_risk_next'] == 1)
).sum()

print("Total Transition Events:", transition_events)

Total Transition Events: 0


In [19]:
df_model['target'] = df_model['riskindx'].shift(-1)
df_model.head()

,transaction_count,riskindx,high_risk_next,target
0,105,15.803059,1.0,14.729404
1,172,14.729404,1.0,14.015591
2,176,14.015591,1.0,12.111834
3,137,12.111834,1.0,10.558757
4,174,10.558757,1.0,10.389005


In [20]:
df_model.tail()
df_model.shape

(316, 4)

In [21]:
df_model = df_model.drop(columns = ['high_risk_next'])
df_model.columns

Index(['transaction_count', 'riskindx', 'target'], dtype='object')

In [22]:
df_model = df_model.dropna(subset = ['target'])
df_model.shape

(315, 3)

In [23]:
splite_index = int(len(df_model)*.7)

train = df_model.iloc[:splite_index]
test = df_model.iloc[splite_index:]

X_train = train[['transaction_count','riskindx']]
y_train = train[['target']]

X_test = test[['transaction_count','riskindx']]
y_test = test[['target']]

In [24]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

In [25]:
model = LinearRegression()
model.fit(X_train,y_train)
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

In [26]:
r2_train = r2_score(y_train,y_train_pred)
r2_test = r2_score(y_test,y_test_pred)

print("R2 train:",r2_train)
print("R2 test:",r2_test)

R2 train: 0.99629020499664
R2 test: 0.8384699807055305


In [27]:
mae = mean_absolute_error(y_test, y_test_pred)
print("MAE:", mae)

MAE: 0.0646596621332756


In [28]:
rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
print("RMSE:", rmse)

RMSE: 0.07625077962595257


In [29]:
coeff = pd.DataFrame({
    'Feature': X_train.columns,
    'Coefficient': model.coef_.flatten()
})
print('Intercept:', model.intercept_)
print(coeff)

Intercept: [0.15738902]
             Feature  Coefficient
0  transaction_count    -0.000027
1           riskindx     0.924571
